# Run Threshold Experiments

Group A – Baseline Quality Filtering<br>
* A1 – No filtering: All available pixels retained.
* A2 – Bad pixels removed.
* A3 – Bad + degraded pixels removed.
* A4 – Bad + degraded + suspicious pixels removed (most restrictive).

Group B – Alternative Filtering Approaches
* B1 – Default filtering: Using the general quality flag.
* B2 – Bitwise filtering: Bad pixels were removed ignoring the no_pixels bitwise flag.
* B3 – Geometric inner-swath exclusion: Pixels inside ±10 km of nadir removed by a geometric approach, instead of using the inner_swath flag.
* B4 – Combined filtering: Applying bitwise and geometric filters jointly.


## Imports

In [1]:
%load_ext autoreload
%autoreload 2

from tqdm.auto import tqdm

from swot_toolkit.analysis import open_sites_and_dates
from swot_toolkit.pipe2 import open_output_dir
from swot_toolkit.pipe4 import calc_swot_metrics, quality_flags_bad


## Define scenarios

In [2]:
scenarios_B = {
    "Default filtering": {
        "exclude_flags": {3},
        "exclude_bitwise": None,
        "exclude_geometric": False,
    },
    "Bitwise filtering": {
        "exclude_flags": set(),
        "exclude_bitwise": set(quality_flags_bad) - {"no_pixels"},
        "exclude_geometric": False,
    },
    "Geometric filtering": {
        "exclude_flags": set(),
        "exclude_bitwise": set(quality_flags_bad) - {"inner_swath"},
        "exclude_geometric": True,
    },
    "Bitwise+Geometric": {
        "exclude_flags": set(),
        "exclude_bitwise": set(quality_flags_bad) - {"no_pixels", "inner_swath"},
        "exclude_geometric": True,
    },
}


scenarios_A = {
    "No filtering": {"exclude_flags": None, "exclude_bitwise": None, "exclude_geometric": False},
    "Bad removed": {"exclude_flags": set()},  # Handled in scenario B
    "Bad + Deg removed": {"exclude_flags": {2}},
    "Bad + Deg + Susp removed": {"exclude_flags": {1, 2}},
}


def get_filtering_params(scenario_A, scenario_B) -> dict:
    if scenario_A == "No filtering":
        return scenarios_A[scenario_A]

    else:
        params = scenarios_B[scenario_B].copy()
        params["exclude_flags"] = (
            params["exclude_flags"] | scenarios_A[scenario_A]["exclude_flags"]
        )
        return params


In [3]:
for i, scenario_A in enumerate(scenarios_A.keys()):
    for j, scenario_B in enumerate(scenarios_B.keys()):
        experiment_name = f"A{i + 1}B{j + 1}"
        filtering_params = get_filtering_params(scenario_A, scenario_B)

        print(f"{experiment_name} - {scenario_A}/{scenario_B}: {filtering_params}")

A1B1 - No filtering/Default filtering: {'exclude_flags': None, 'exclude_bitwise': None, 'exclude_geometric': False}
A1B2 - No filtering/Bitwise filtering: {'exclude_flags': None, 'exclude_bitwise': None, 'exclude_geometric': False}
A1B3 - No filtering/Geometric filtering: {'exclude_flags': None, 'exclude_bitwise': None, 'exclude_geometric': False}
A1B4 - No filtering/Bitwise+Geometric: {'exclude_flags': None, 'exclude_bitwise': None, 'exclude_geometric': False}
A2B1 - Bad removed/Default filtering: {'exclude_flags': {3}, 'exclude_bitwise': None, 'exclude_geometric': False}
A2B2 - Bad removed/Bitwise filtering: {'exclude_flags': set(), 'exclude_bitwise': {'outside_data_window', 'inner_swath', 'missing_karin_data', 'outside_scene_bounds', 'value_bad'}, 'exclude_geometric': False}
A2B3 - Bad removed/Geometric filtering: {'exclude_flags': set(), 'exclude_bitwise': {'outside_data_window', 'missing_karin_data', 'no_pixels', 'outside_scene_bounds', 'value_bad'}, 'exclude_geometric': True}
A2B

## Run experiments

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from swot_toolkit.metrics import calc_metrics, process_swot_mask
from swot_toolkit.pipe2 import open_output_dir, open_roi
from swot_toolkit.pipe4 import open_datasets
from swot_toolkit.swot import create_raster_mosaic_combined

metrics = ["iou", "f1", "precision", "recall"]

sites_dates = open_sites_and_dates("/data/swot/output")
sites_dates


{'Curua-Una': ['2024-07-13', '2025-08-14'],
 'Northeast': ['2024-05-29', '2025-07-20'],
 'Rio_Branco': ['2024-04-03', '2025-09-07'],
 'Rio_Madeira': ['2024-08-21', '2025-07-21'],
 'Rio_Negro': ['2024-11-29', '2025-08-07']}

In [22]:
params = get_filtering_params("Bad removed", "Bitwise filtering")


for region, dates in tqdm(sites_dates.items(), desc="Regions"):
    # Open basic info about the region
    base_dir, aoi, mosaic_df = open_roi(region)

    for date in dates:
        # Open datasets for the specific region and date
        datasets = open_datasets(region, date)

        raster, patches, no_data_masks = create_raster_mosaic_combined(
            mosaic_df=mosaic_df,
            ref_date=date,
            aoi=aoi,
            **params,
        )

        metrics_df = pd.DataFrame()

        # Noe, let's loop through the thresholds
        for thresh in np.arange(0.1, 1, 0.01):
            pred_mask = process_swot_mask(raster, water_threshold=thresh)

            stats = calc_metrics(
                datasets["ref_mask"],
                pred_mask,
                metrics=metrics,
                binary=True,
            )
            stats = stats.rename(columns={0: f">{thresh:.2f}"})

            metrics_df = pd.concat([metrics_df, stats], axis=1)

        # After looping through all experiments, save the metrics for this region and date
        output_path = base_dir / date / "results"
        output_path.mkdir(parents=True, exist_ok=True)
        metrics_df.to_parquet(output_path / f"{region}_{date}_thresholds.parquet")



Regions:   0%|          | 0/5 [00:00<?, ?it/s]

The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2', 'opera_s1']
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2', 'opera_s1']
Reading KML file: /data/swot/output/Rio_Branco/kml/Rio_Branco.kml
No OPERA S1 mask available for date 202403
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2']
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2', 'opera_s1']
Reading KML file: /data/swot/output/Rio_Madeira/kml/Rio_Madeira.kml
No OPERA S1 mask available for date 202408
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2']
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2', 'opera_s1']
Reading KML file: /data/swot/output/Rio_Negro/kml/Rio_Negro.kml
No OPERA S1 mask available for date 202411
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2']
The following datasets have been opened: ['s

In [23]:
metrics_df.iloc[:, 40:60]

,>0.50,>0.51,>0.52,>0.53,>0.54,>0.55,>0.56,>0.57,>0.58,>0.59,>0.60,>0.61,>0.62,>0.63,>0.64,>0.65,>0.66,>0.67,>0.68,>0.69
iou,0.7460,0.7890,0.7901,0.7907,0.7901,0.7894,0.7881,0.7867,0.7854,0.7832,0.7816,0.7793,0.7765,0.7735,0.7709,0.7678,0.7649,0.7610,0.7578,0.7537
f1,0.8545,0.8821,0.8827,0.8831,0.8828,0.8823,0.8815,0.8806,0.8798,0.8784,0.8774,0.8760,0.8742,0.8723,0.8706,0.8687,0.8668,0.8643,0.8622,0.8595
precision,0.7986,0.8614,0.8669,0.8725,0.8760,0.8792,0.8822,0.8849,0.8880,0.8904,0.8929,0.8951,0.8972,0.8995,0.9019,0.9040,0.9060,0.9079,0.9104,0.9122
recall,0.9189,0.9038,0.8991,0.8940,0.8897,0.8854,0.8808,0.8764,0.8718,0.8668,0.8624,0.8577,0.8522,0.8466,0.8414,0.8360,0.8308,0.8247,0.8189,0.8127
